# <font color="purple">**Greek ERGANI XML Data Validation Tool**</font>
This notebook breaks down the pre-submission auditing tool used to verify employee daily work schedule XML against ERGANI compliance standards. It is designed to log every single missing data element alongside its exact physical line number in the XML file.

### <font color="blue">**Step 1: Load Dependencies**</font>
The first step is to import the necessary standard modules for handling web requests, parsing XML structures and using regular expressions (`urllib`, `ElementTree`, `re`).

In [22]:
import urllib.request, urllib.parse, urllib.error
import xml.etree.ElementTree as ET
import re

### <font color="blue"> **Step 2: Fetch & Decode Raw Data**</font>
After requesting the user to provide the URL of the raw XML file, urllib is used to open, read and decode the file, and then, using `.splitlines()`, the file is converted to a list of elements for future processing. Each element of the list is a line of the XML file.

<font color="red">**NOTE:**</font> If the user does not provide the Python tool with a URL, the file 'Work Schedule 22-23.05.2026.xml' will be used by default.

In [23]:
# Request the  raw XML file.
url=input('Enter the URL of your raw XML file (or press ENTER for default): ')
if len(url)<1:
   url='https://raw.githubusercontent.com/kyreugenia/Data_Analysis_Portfolio/refs/heads/main/ERGANI_XML_Data%20Validator/Work%20Schedule%2022-23.05.2026.xml'

# Open and read the XML file as a list of plain lines.
link=urllib.request.urlopen(url)
line_list = link.read().decode().splitlines()
print (f'\033[1;95mFor demonstration purposes, the first three elements of the produced list will look like this:\033[0m\n{line_list[:3]}')

For demonstration purposes, the first three elements of the produced list will look like this:
['<?xml version="1.0" encoding="utf-8"?>', '<WTOS xmlns:xsd="http://www.w3.org/2001/XMLSchema" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns="http://www.yeka.gr/WTO">', '  <WTO xmlns="">']


### <font color="blue"> **Step 3: Inject Line Numbers**</font>
This step focuses on determining the line of every key tag in the XML file. 

Noticing that all the key tags have a predictable pattern (starting  with  `f_` and closing with `>`), it makes it easy to locate them using `regular expressions`.

Then, leveraging the `re.sub()` function, the script automatically replaces the standard opening tag with an updated version that embeds a custom `line_num` attribute.  

Once the process is complete, the updated `line_list` is concatenated into a valid XML string format (by joining the lines sequentially using `\n`) and then parsed into the XML tree using the `fromstring()` function of **ElementTree**.

In [24]:
# Add line number attribute.
for line in range(len(line_list)):
   if '<f_' in line_list[line]:
      line_list[line]=re.sub(r'<f_(\w+)>', r'<f_\1 line_num="' + str(line+1) + '">', line_list[line])

# Parse the modified text into the XML tree.
tree = ET.fromstring("\n".join(line_list))

### <font color="blue"> **Step 4: Find & Map Key Tags**</font>
During this step, the `.findall()` function locates all the key tags nested under `ErgazomenoiWTO` that need compliance checking.    

A dictionary is also defined to map the raw XML tags to user-friendly reporting names. These names will be shown later on the report. (e.g. If a 'f_eponymo' tag is empty, the program will print "Missing surname in line xx" instead of "Missing f_eponymo in line xx") 

In [25]:
element_list=tree.findall('.//ErgazomenoiWTO//')

# Map tags to user-friendly names for reporting.
tags= {'f_afm':'AFM','f_eponymo':'surname','f_onoma':'name','f_date':'date','f_type':'type','f_from': 'start hour','f_to':'end hour'}

### <font color="blue"> **Step 5: Execute Audit Loop**</font>
This step runs the final check. It mainly focuses on locating empty key tags and generating the final validation report, incorporating the **Line Numbers** that were defined in <font color="dodgerblue"> **Step 3**</font> and the **Tags Dictionary** from <font color="dodgerblue"> **Step 4**</font>.    

During this step, the below points must be considered:
* <font color="orchid">**Structural Exclusions:**</font>`ErgazomenosAnalytics` container tag and its child tag, `ErgazomenosWTOAnalytics`, are both nested under the `ErgazomenoiWTO` tag. Because they serve as organizational wrappers rather than data fields requiring verification, the tool is programmed to skip them entirely.
* <font color="orchid">**Time Tags Check:**</font> If the "f_type" is flagged as **"ΑΝ"** (represents a Rest Day), it is expected for "f_from" (Start Hour) and "f_to" (End Hour) tags to be empty.
* <font color="orchid">**AFM Check:**</font> In addition to identifying missing AFM numbers, the tool enforces a strict length constraint. All AFM numbers must consist of exactly 9 digits to be considered valid under Greek regulatory standards.

In [ ]:
for index,element in enumerate(element_list):
   # Skip the tags 'ErgazomenosAnalytics' and 'ErgazomenosWTOAnalytics'.
   if element.tag=='ErgazomenosAnalytics'or element.tag=='ErgazomenosWTOAnalytics':
      continue

   # Flag any data found on Rest Days ("ΑΝ").
   if element.tag=='f_from' and (element_list[index-1]).text=="ΑΝ":
      if element.text:
         print('Start Hour in line',element.get('line_num'),'is expected to be blank for Rest Days')
      continue
   if element.tag=='f_to' and (element_list[index-2]).text=="ΑΝ":
      if element.text:
         print('End Hour in line',element.get('line_num'),'is expected to be blank for Rest Days')
      continue

   #Check for missing elements.
   if element.text is None or element.text.strip()=='':
      print ('Missing',tags[element.tag],'in line',element.get('line_num'))
      continue

   # Verify AFM length.
   if element.tag=='f_afm': 
      if element.text is not None and len(element.text)!=9: 
         print('Wrong AFM in line',element.get('line_num'))

Wrong AFM in line 12
Wrong AFM in line 25
Start Hour in line 32 is expected to be blank for Rest Days
End Hour in line 33 is expected to be blank for Rest Days
Missing name in line 95
Missing date in line 109
Missing AFM in line 120
Missing type in line 139
Missing start hour in line 140
Missing end hour in line 141
Missing surname in line 161
Wrong AFM in line 471
Missing type in line 1261
Missing start hour in line 1262
Missing end hour in line 1263
